# Notebook 01 — Data Loading & Experiment Validation
**Dataset:** A/B Testing Dataset (Kaggle) — Landing Page Conversion  
**Tujuan:** Validasi kualitas eksperimen sebelum analisis statistik.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../output')
FIGURES = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

BLUE = '#2563EB'
RED  = '#DC2626'
GRAY = '#6B7280'
sns.set_theme(style='whitegrid', font_scale=1.1)

## 1. Load Data

In [ ]:
df = pd.read_csv(RAW / 'ab_data.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
print(f'\nKolom: {df.columns.tolist()}')
print(f'\nDistribusi grup:')
print(df['group'].value_counts())
df.head()

## 2. Validasi Kualitas Eksperimen

> Sebelum analisis statistik, kita WAJIB memvalidasi bahwa eksperimen berjalan dengan benar.
> Kesalahan assignment merusak validitas seluruh eksperimen.

In [ ]:
print('=== VALIDASI EKSPERIMEN ===')

# 1. Duplikat user_id
dup_users = df[df.duplicated('user_id', keep=False)]
print(f'\n[1] User dengan lebih dari 1 baris : {df["user_id"].duplicated().sum():,}')

# 2. User di kedua grup
users_per_group = df.groupby('user_id')['group'].nunique()
users_two_groups = users_per_group[users_per_group > 1]
print(f'[2] User di KEDUA grup (assignment error): {len(users_two_groups):,}')

# 3. Mismatch group ↔ landing_page
mismatch = df[
    ((df['group'] == 'control')   & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]
print(f'[3] Baris mismatch group ↔ landing_page: {len(mismatch):,}')

print(f'\nTotal baris sebelum cleaning: {len(df):,}')

In [ ]:
# Cleaning: hapus mismatch → sort timestamp → hapus duplikat user (ambil pertama)
df_clean = df[
    ((df['group'] == 'control')   & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
].copy()

df_clean = (
    df_clean
    .sort_values('timestamp')
    .drop_duplicates('user_id', keep='first')
    .reset_index(drop=True)
)

print(f'Setelah cleaning: {len(df_clean):,} baris')
print(f'User unik: {df_clean["user_id"].nunique():,}')
print('\nDistribusi setelah cleaning:')
print(df_clean['group'].value_counts())

## 3. Statistik Dasar Eksperimen

In [ ]:
summary = (
    df_clean.groupby('group')
    .agg(
        n           = ('user_id', 'count'),
        conversions = ('converted', 'sum'),
        conv_rate   = ('converted', 'mean')
    )
    .round(4)
)

ctrl  = summary.loc['control']
treat = summary.loc['treatment']
lift  = (treat['conv_rate'] - ctrl['conv_rate']) / ctrl['conv_rate'] * 100

print('=== RINGKASAN EKSPERIMEN ===')
print(summary)
print(f'\nAbsolute lift  : {treat["conv_rate"] - ctrl["conv_rate"]:+.4f}')
print(f'Relative lift  : {lift:+.2f}%')
print(f'Durasi         : {df_clean["timestamp"].min().date()} → {df_clean["timestamp"].max().date()}')

## 4. Distribusi Temporal — Cek Novelty Effect

In [ ]:
df_clean['date'] = df_clean['timestamp'].dt.date
daily = (
    df_clean.groupby(['date', 'group'])['converted']
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Daily traffic
traffic = df_clean.groupby(['date', 'group'])['user_id'].count().reset_index()
for grp, color in [('control', BLUE), ('treatment', RED)]:
    d = traffic[traffic['group'] == grp]
    axes[0].plot(d['date'], d['user_id'], color=color, linewidth=1.5, label=grp)
axes[0].set_title('Daily Traffic per Grup', fontweight='bold')
axes[0].set_ylabel('Pengguna per Hari')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Daily conversion rate
for grp, color in [('control', BLUE), ('treatment', RED)]:
    d = daily[daily['group'] == grp]
    axes[1].plot(d['date'], d['converted'] * 100, color=color, linewidth=1.5, label=grp)
axes[1].set_title('Daily Conversion Rate per Grup\n(cek novelty effect)', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES / 'A_temporal_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: A_temporal_check.png')

## 5. Generate Revenue Column & Simpan

In [ ]:
np.random.seed(42)
n = len(df_clean)

# Revenue: 0 jika tidak konversi, Normal(45,15) jika konversi
raw_revenue = np.random.normal(loc=45, scale=15, size=n).clip(min=5)
df_clean['revenue'] = np.where(df_clean['converted'] == 1, raw_revenue, 0.0)

print('Revenue stats per grup:')
print(df_clean.groupby('group')['revenue'].agg(['mean','median','std']).round(3))

df_clean.to_parquet(OUT / 'df_clean.parquet', index=False)
print(f'\nTersimpan: df_clean.parquet — {len(df_clean):,} baris')